In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA_ROOT = Path("../data/sample/processed")

# ============================================================================
# Load Stage 4 Outputs
# ============================================================================

instance_labels = np.load(
    DATA_ROOT /
    "stage_4_instance_segmentation" /
    "instance_labels.npy"
)

# ============================================================================
# Load Stage 5 Outputs
# ============================================================================

cells_df = pd.read_csv(
    DATA_ROOT /
    "stage_5_cell_detection" /
    "cells.csv"
)

print("Instance Labels")
print("----------------")
print("Shape :", instance_labels.shape)
print("Dtype :", instance_labels.dtype)
print("Cells :", instance_labels.max())

print()

print(f"Detected cells : {len(cells_df)}")

display(cells_df.head())

Instance Labels
----------------
Shape : (64, 256, 256)
Dtype : int32
Cells : 199

Detected cells : 199


,cell_id,centroid_z,centroid_y,centroid_x,volume_voxels,z_min,y_min,x_min,z_max,y_max,x_max
0,1,0.231183,6.354839,52.204301,186.0,0,0,46,2,14,60
1,2,0.153153,6.815315,71.774775,222.0,0,0,65,2,16,80
2,3,0.320833,26.012500,65.191667,240.0,0,18,59,4,35,72
3,4,1.074627,62.743555,58.230665,737.0,0,55,48,4,72,68
4,5,0.364431,98.571429,34.836735,343.0,0,89,28,2,109,43


In [2]:
import zarr

PROJECT_ROOT = Path.cwd().parent

DATA_ROOT = (
        PROJECT_ROOT
        / "data" / "sample"
        / "biohub_5samples_20timepoints"
        / "train"
)

SAMPLE_ID = "44b6_0113de3b"

ZARR_PATH = (
        DATA_ROOT
        / SAMPLE_ID
        / f"{SAMPLE_ID}.zarr"
)

print(ZARR_PATH)
print(ZARR_PATH.exists())

ARRAY_PATH = ZARR_PATH / "0"

volume = zarr.open_array(
    str(ARRAY_PATH),
    mode="r"
)


D:\Projects\Kaggle\cell-tracking\data\sample\biohub_5samples_20timepoints\train\44b6_0113de3b\44b6_0113de3b.zarr
True


In [3]:
# ============================================================================
# Load Original Timepoint
# ============================================================================

TIMEPOINT = 0

original_volume = volume[TIMEPOINT]

print("Original Volume")
print("---------------")
print("Shape :", original_volume.shape)
print("Dtype :", original_volume.dtype)

Original Volume
---------------
Shape : (64, 256, 256)
Dtype : uint16


Extract Intensity Features

In [4]:
# ============================================================================
# Feature Extraction
# ============================================================================

feature_rows = []

for cell_id in cells_df["cell_id"]:

    # Voxels belonging to this cell
    mask = instance_labels == cell_id

    # Intensity values from the ORIGINAL image
    intensities = original_volume[mask]

    feature_rows.append({
        "cell_id": cell_id,
        "intensity_mean": intensities.mean(),
        "intensity_median": np.median(intensities),
        "intensity_std": intensities.std(),
        "intensity_min": intensities.min(),
        "intensity_max": intensities.max(),
        "intensity_q25": np.percentile(intensities, 25),
        "intensity_q75": np.percentile(intensities, 75),
    })

features_df = pd.DataFrame(feature_rows)

# Merge with cell detections
cells_df = cells_df.merge(features_df, on="cell_id")

print(f"Extracted features for {len(cells_df)} cells")

display(cells_df.head())

Extracted features for 199 cells


,cell_id,centroid_z,centroid_y,centroid_x,volume_voxels,z_min,y_min,x_min,z_max,y_max,x_max,intensity_mean,intensity_median,intensity_std,intensity_min,intensity_max,intensity_q25,intensity_q75
0,1,0.231183,6.354839,52.204301,186.0,0,0,46,2,14,60,1028.021505,941.0,296.678929,587,1792,809.25,1182.00
1,2,0.153153,6.815315,71.774775,222.0,0,0,65,2,16,80,1112.540541,1102.5,183.527683,773,1534,950.50,1247.50
2,3,0.320833,26.012500,65.191667,240.0,0,18,59,4,35,72,1133.070833,1026.0,285.671969,729,1744,893.75,1363.75
3,4,1.074627,62.743555,58.230665,737.0,0,55,48,4,72,68,1382.869742,1344.0,433.686663,601,2397,1000.00,1726.00
4,5,0.364431,98.571429,34.836735,343.0,0,89,28,2,109,43,944.402332,925.0,208.344722,582,1407,780.00,1102.00
